# Case 3 — SCD Type 2

## 1. Create sample customer changes
Customer 1 has a late-arriving change; customer 2 changes twice in one day; customer 3 disappears from the next complete snapshot.

In [0]:
from pyspark.sql import functions as F, Window
from datetime import datetime

rows = [
    (1, "Ana", "Toronto", datetime(2026,11,1,9), 1, "U"),
    (1, "Ana", "Windsor", datetime(2026,11,10,9), 3, "U"),
    (1, "Ana", "Ottawa", datetime(2026,11,5,9), 2, "U"), # arrived late
    (2, "Ben", "Toronto", datetime(2026,11,3,8), 4, "U"),
    (2, "Ben", "Ottawa", datetime(2026,11,3,12), 5, "U"),
    (2, "Ben", "Windsor", datetime(2026,11,3,17), 6, "U"),
    (3, "Cara", "London", datetime(2026,11,1,9), 7, "U"),
]
columns = ["customer_id", "customer_name", "city", "effective_at", "event_seq", "operation"]
events = spark.createDataFrame(rows, columns)
display(events.orderBy("customer_id", "effective_at"))

## 2. Find customers missing from a complete snapshot
A complete November 10 snapshot contains customers 1 and 2, but not customer 3. We generate a delete/closure event for customer 3.

In [0]:
previous_snapshot = spark.createDataFrame([(1,), (2,), (3,)], ["customer_id"])
today_snapshot = spark.createDataFrame([(1,), (2,)], ["customer_id"])
missing = previous_snapshot.join(today_snapshot, "customer_id", "left_anti")
deletes = (missing
    .withColumn("customer_name", F.lit(None).cast("string"))
    .withColumn("city", F.lit(None).cast("string"))
    .withColumn("effective_at", F.to_timestamp(F.lit("2026-11-10 23:59:00")))
    .withColumn("event_seq", F.lit(8).cast("long"))
    .withColumn("operation", F.lit("D"))
    .select(*columns))
all_events = events.unionByName(deletes)
display(all_events.orderBy("customer_id", "effective_at"))

## 3. Build SCD Type 2 history
Deduplicate exact repeated events; keep the latest event when timestamps are identical; close the previous row at the next change. `effective_to` is exclusive.

In [0]:
def build_scd2(changes):
    changes = changes.select(*columns).dropDuplicates()
    # Never silently choose between conflicting events with the same ordering keys.
    conflicts = (changes.groupBy("customer_id", "effective_at", "event_seq")
                 .count().filter("count > 1").limit(1).count())
    if conflicts:
        raise ValueError("Conflicting events with identical timestamp and sequence")

    same_time = Window.partitionBy("customer_id", "effective_at").orderBy(F.desc("event_seq"))
    changes = (changes.withColumn("rn", F.row_number().over(same_time))
               .filter("rn = 1").drop("rn"))

    order = Window.partitionBy("customer_id").orderBy("effective_at", "event_seq")
    changes = (changes.withColumn("previous_city", F.lag("city").over(order))
               .withColumn("previous_name", F.lag("customer_name").over(order))
               .withColumn("previous_operation", F.lag("operation").over(order)))
    changes = changes.filter(
        F.col("previous_operation").isNull()
        | (F.col("operation") != F.col("previous_operation"))
        | ~F.col("city").eqNullSafe(F.col("previous_city"))
        | ~F.col("customer_name").eqNullSafe(F.col("previous_name"))
    )
    return (changes.withColumn("effective_to", F.lead("effective_at").over(order))
            .filter("operation = 'U'")
            .select("customer_id", "customer_name", "city",
                    F.col("effective_at").alias("effective_from"), "effective_to",
                    F.col("effective_to").isNull().alias("is_current")))

history = build_scd2(all_events)
display(history.orderBy("customer_id", "effective_from"))

## 4. Check results

In [0]:
assert history.filter("customer_id = 1").count() == 3  # late change placed in history
assert history.filter("customer_id = 2").count() == 3  # two changes in one day
assert history.filter("customer_id = 3 AND is_current = true").count() == 0  # disappeared
assert history.filter("is_current = true").count() == 2
assert history.groupBy("customer_id").agg(F.sum(F.col("is_current").cast("int")).alias("n")).filter("n > 1").count() == 0
print("All demonstration checks passed")